# Local July ST Circulant Simulation Generator — smooth=1.0

Moved-location local runner for:

```text
/Users/joonwonlee/Documents/GEMS_TCO-1/simulate_data/generate_july_st_circulant_real_locations_2022_2025_smooth0p3_051926.py
```

This notebook writes generated simulation assets outside the git repo under:

```text
/Users/joonwonlee/Documents/GEMS_DATA/simulation/july_st_circulant_realpattern_smooth1p0
```

It is intentionally **not auto-running** because the full July 2022-2025 generation is slow.
Set `RUN_NOW = True` only after checking paths and output directory.

Smooth-specific behavior:
- `smooth=0.5`: closed-form exponential Matérn correlation inside the generator
- `smooth=0.3` or `smooth=1.0`: natural cubic spline Matérn correlation


In [ ]:
import os
import sys
import shlex
import subprocess
from pathlib import Path

ROOT = Path('/Users/joonwonlee/Documents/GEMS_TCO-1')
GENERATOR = ROOT / 'simulate_data' / 'generate_july_st_circulant_real_locations_2022_2025_smooth0p3_051926.py'
INPUT_ROOT = Path('/Users/joonwonlee/Documents/GEMS_DATA')
OUTPUT_DIR = Path('/Users/joonwonlee/Documents/GEMS_DATA') / 'simulation' / 'july_st_circulant_realpattern_smooth1p0'

SMOOTH = 1.0
YEARS = '2022,2023,2024,2025'
MAX_HOURS = 248
HOURS_PER_DAY = 8
SEED = 20240701

# Full run is slow. For a quick local smoke test, set YEARS='2024' and MAX_HOURS=8.
RUN_NOW = False

assert GENERATOR.exists(), f'Missing generator: {GENERATOR}'
print('Generator:', GENERATOR)
print('Input root:', INPUT_ROOT, 'exists=', INPUT_ROOT.exists())
print('Output dir:', OUTPUT_DIR)
print('Smooth:', SMOOTH)


In [ ]:
expected_inputs = [
    INPUT_ROOT / 'pickle_2022' / 'tco_grid_22_07.pkl',
    INPUT_ROOT / 'pickle_2023' / 'tco_grid_23_07.pkl',
    INPUT_ROOT / 'pickle_2024' / 'tco_grid_24_07.pkl',
    INPUT_ROOT / 'pickle_2025' / 'tco_grid_25_07.pkl',
]
for p in expected_inputs:
    print(('OK   ' if p.exists() else 'MISS '), p)


## Build command

The command writes one output folder per year and creates both real-location and gridded pickles.


In [ ]:
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
os.environ.setdefault('PYTHONPATH', str(ROOT / 'src'))
os.environ.setdefault('OMP_NUM_THREADS', '4')
os.environ.setdefault('MKL_NUM_THREADS', '4')
os.environ.setdefault('OPENBLAS_NUM_THREADS', '4')
os.environ.setdefault('NUMEXPR_NUM_THREADS', '4')

cmd = [
    sys.executable, str(GENERATOR),
    '--years', YEARS,
    '--input-root', str(INPUT_ROOT),
    '--output-dir', str(OUTPUT_DIR),
    '--max-hours', str(MAX_HOURS),
    '--hours-per-day', str(HOURS_PER_DAY),
    '--seed', str(SEED),
    '--smooth', str(SMOOTH),
    '--spline-n-points', '4000',
    '--spline-r-max', '30.0',
    '--sigmasq', '10.0',
    '--range-lat', '0.2',
    '--range-lon', '0.3',
    '--range-time', '2.0',
    '--advec-lat', '0.08',
    '--advec-lon', '-0.2',
    '--nugget', '1.0',
    '--mean-intercept', '260.0',
    '--mean-lat-slope', '1.0',
    '--mean-lat-center', '-0.5',
    '--lat-range=-3,2',
    '--lon-range=121,131',
    '--lat-factor-hr', '100',
    '--lon-factor-hr', '10',
    '--hr-pad', '0.1',
]
print(' '.join(shlex.quote(x) for x in cmd))


In [ ]:
if RUN_NOW:
    subprocess.run(cmd, check=True)
else:
    print('RUN_NOW is False. Set RUN_NOW=True to generate data.')


## Verify outputs


In [ ]:
if OUTPUT_DIR.exists():
    for year_dir in sorted(OUTPUT_DIR.glob('*_july_st_circulant')):
        files = sorted(p.name for p in year_dir.iterdir())
        print()
        print(year_dir)
        for name in files:
            print('  ', name)
else:
    print('No output directory yet:', OUTPUT_DIR)
